<a href="https://colab.research.google.com/github/amit-dhidhi-dev/google_colab/blob/main/book_summary_upload_channel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **step 1: start creating video**

In [ ]:
# @title mount google drive
from google.colab import drive
drive.mount('/content/drive')

# **Upload all files**

In [ ]:
# @title upload story (.txt)
from google.colab import files
from pathlib import Path
print("कृपया  story फाइल अपलोड करें (txt )")
text_file = files.upload()
filename = list(text_file.keys())[0] if text_file else None
story_file_path = Path(filename)

# Saara text read karo
text_to_speak = story_file_path.read_text(encoding='utf-8')

# print(text_to_speak)

In [ ]:
# @title upload  video details file (.json)
from google.colab import files
from pathlib import Path
print("कृपया  video details फाइल अपलोड करें (txt )")
video_details = files.upload()
videodetails_file = list(video_details.keys())[0] if video_details else None
video_details_file_path = Path(videodetails_file)

In [ ]:
# @title upload shorts story (.txt)
from google.colab import files
from pathlib import Path

print("कृपया shorts story फ़ाइल अपलोड करें (txt)")
shorts_text_file = files.upload()
shorts_filename = list(shorts_text_file.keys())[0] if shorts_text_file else None
shorts_story_file_path = Path(shorts_filename)

# Saara text read karo
shorts_text_to_speak = shorts_story_file_path.read_text(encoding='utf-8')

print(f"Shorts story loaded from: {shorts_story_file_path}")

In [ ]:
# @title upload shorts video details file (.json)

print("कृपया shorts video details फ़ाइल अपलोड करें (JSON)")
shorts_video_details_upload = files.upload()
shorts_videodetails_file = list(shorts_video_details_upload.keys())[0] if shorts_video_details_upload else None
shorts_video_details_file_path = Path(shorts_videodetails_file)

import json
with open(shorts_video_details_file_path, "r", encoding="utf-8") as f:
    shorts_video_details = json.load(f)

print(f"Shorts video details loaded from: {shorts_video_details_file_path}")

In [ ]:
# @title Upload Shorts Background Image
from google.colab import files
from pathlib import Path

print("कृपया shorts background image फाइल अपलोड करें")
shorts_img_upload = files.upload()
shorts_bg_image_file = list(shorts_img_upload.keys())[0] if shorts_img_upload else None
shorts_bg_image_file_path = Path(shorts_bg_image_file)

print(f"Shorts background image loaded from: {shorts_bg_image_file_path}")

In [ ]:
# @title upload single image
print("कृपया  image फाइल अपलोड करें work both for bg and thumbnail")
thumbnail_file = files.upload()
thumbnail_image_file = list(thumbnail_file.keys())[0] if thumbnail_file else None
thumbnail_image_file_path = Path(thumbnail_image_file)
bg_image_file = thumbnail_image_file
bg_image_file_path = thumbnail_image_file_path

In [ ]:
# @title install edge-tts
!pip install edge-tts

In [ ]:
# @title create audio file (story.mp3)
import edge_tts

TEXT = text_to_speak
VOICE = "hi-IN-SwaraNeural"
OUTPUT_FILE = "story.mp3"

# रफ़्तार (Speed), पिच (Pitch), और वॉल्यूम (Volume) सेट करना
SPEED = "+20%"    # रफ़्तार
PITCH = "+0Hz"    # आवाज़ का बेस/पतलापन (e.g., '+20Hz' for high pitch, '-20Hz' for deep voice)
VOLUME = "+200%"    # आवाज़ की तेज़ी

async def main():
    # यहाँ सभी पैरामीटर्स एक साथ दिए गए हैं
    communicate = edge_tts.Communicate(
        TEXT,
        VOICE,
        rate=SPEED,
        pitch=PITCH,
        volume=VOLUME
    )
    await communicate.save(OUTPUT_FILE)

await main()
print(f"Audio updated with Speed: {SPEED}, Pitch: {PITCH}, Volume: {VOLUME}")

In [ ]:
# @title create video (final_story_video.mp4)
import numpy as np
import cv2
from moviepy.editor import *
import moviepy.audio.fx.all as afx
from PIL import Image, ImageFilter
import os
import tempfile
import random
from tqdm.auto import tqdm

class ProfessionalVoiceVideo:
    def __init__(self, image_path, audio_path, bg_music_path=None, output_path="final_story_video.mp4"):
        self.image_path = image_path
        self.audio_path = audio_path
        self.bg_music_path = bg_music_path
        self.output_path = output_path

        audio_clip = AudioFileClip(audio_path)
        self.sample_rate = audio_clip.fps
        self.duration = audio_clip.duration
        chunks = list(audio_clip.iter_chunks(fps=self.sample_rate, chunksize=100000))
        self.audio_data = np.concatenate(chunks)
        if len(self.audio_data.shape) > 1:
            self.audio_data = np.mean(self.audio_data, axis=1)
        audio_clip.close()

        self.original_image = cv2.cvtColor(np.array(Image.open(image_path).convert('RGB')), cv2.COLOR_RGB2BGR)
        self.particles = [{'x': random.random(), 'y': random.random(), 's': random.random()} for _ in range(50)]

    def get_smooth_amplitude(self, duration, total_frames):
        samples_per_frame = len(self.audio_data) // total_frames
        envelope = []
        for i in range(total_frames):
            start, end = i * samples_per_frame, (i + 1) * samples_per_frame
            amp = np.max(np.abs(self.audio_data[start:end])) if start < len(self.audio_data) else 0
            envelope.append(amp)
        if max(envelope) > 0: envelope = np.array(envelope) / max(envelope)
        return np.convolve(envelope, np.ones(5)/5, mode='same')

    def apply_professional_effects(self, img, amp, frame_num, total_frames):
        h, w = img.shape[:2]
        t = frame_num / total_frames
        zoom = 1.0 + 0.1 * np.sin(t * np.pi)
        pan_x = int(np.sin(t * 2) * 20)
        pan_y = int(np.cos(t * 2) * 20)
        M = np.float32([[zoom, 0, (1-zoom)*w/2 + pan_x], [0, zoom, (1-zoom)*h/2 + pan_y]])
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        overlay = img.copy()
        for p in self.particles:
            px, py = int(p['x'] * w), int((p['y'] - (frame_num * 0.001 * p['s'])) % 1 * h)
            size = int(2 + amp * 5 * p['s'])
            cv2.circle(overlay, (px, py), size, (255, 255, 255), -1)
        img = cv2.addWeighted(img, 0.85, overlay, 0.15, 0)
        wave_h, bars = 120, 80
        bar_w = w // bars
        for i in range(bars):
            dist_from_center = 1.0 - abs(i - bars/2) / (bars/2)
            h_val = int(wave_h * amp * dist_from_center * np.random.uniform(0.6, 1.0))
            color = (int(100 + 155*t), int(200 - 100*amp), 255)
            cv2.line(img, (i*bar_w + 2, h-20), (i*bar_w + 2, h-20-h_val), color, 2, cv2.LINE_AA)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.circle(mask, (w//2, h//2), int(max(w,h)*0.8), 255, -1)
        mask = cv2.GaussianBlur(mask, (151, 151), 0) / 255.0
        img = (img * mask[:,:,np.newaxis]).astype(np.uint8)
        return img

    def create_video(self, fps=24, bg_music_volume=0.01):
        total_frames = int(self.duration * fps)
        envelope = self.get_smooth_amplitude(self.duration, total_frames)
        temp_video = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        temp_video.close()
        h, w = self.original_image.shape[:2]
        out = cv2.VideoWriter(temp_video.name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        print("Rendering Visual Frames...")
        for i in tqdm(range(total_frames), desc="Video Rendering"):
            frame = self.apply_professional_effects(self.original_image.copy(), envelope[i], i, total_frames)
            out.write(frame)
        out.release()

        print("Finalizing Audio and Exporting...")
        final_clip = VideoFileClip(temp_video.name)
        voice_audio = AudioFileClip(self.audio_path)
        if self.bg_music_path and os.path.exists(self.bg_music_path):
            bg_audio = AudioFileClip(self.bg_music_path).volumex(bg_music_volume)
            bg_audio = afx.audio_loop(bg_audio, duration=self.duration).audio_fadein(2).audio_fadeout(2)
            final_audio = CompositeAudioClip([voice_audio, bg_audio])
            final_clip = final_clip.set_audio(final_audio)
        else:
            final_clip = final_clip.set_audio(voice_audio)

        # Added logger='bar' to monitor progress
        final_clip.write_videofile(self.output_path, codec='libx264', audio_codec='aac', bitrate="2500k", logger='bar')
        os.unlink(temp_video.name)

def main():
    img, voice = bg_image_file, "story.mp3"
    bg ="/content/drive/MyDrive/readbucks/bg_sound.mp3"
    if os.path.exists(img):
        engine = ProfessionalVoiceVideo(img, voice, bg)
        engine.create_video(fps=24, bg_music_volume=0.01)
        print("Professional Video Rendered Successfully!")

if __name__ == "__main__":
    main()

# **upload video to channel (readbucks)**

In [ ]:
# @title step1. get_youtube_service


import os
import pickle
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.colab import files
import io
from PIL import Image
import requests

# YouTube API scopes (thumbnail upload के लिए)
SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",
    "https://www.googleapis.com/auth/youtube"
]

def get_youtube_service():
    """YouTube API service बनाता है"""
    credentials = None

    # Check if token exists
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            credentials = pickle.load(token)

    # If credentials are invalid, get new ones
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else:
            # Upload credentials file
            print("Google API credentials JSON फ़ाइल अपलोड करें:")
            uploaded = files.upload()
            credentials_file = list(uploaded.keys())[0]

            flow = InstalledAppFlow.from_client_secrets_file(
                credentials_file, SCOPES)
            credentials = flow.run_local_server(port=0)

        # Save credentials
        with open('token.pickle', 'wb') as token:
            pickle.dump(credentials, token)

    return build('youtube', 'v3', credentials=credentials)

In [ ]:
# @title step 2. **थंबनेल अपलोड Function**

def upload_thumbnail(youtube, video_id, thumbnail_path):
    """
    YouTube वीडियो के लिए थंबनेल अपलोड करें

    Parameters:
    - youtube: YouTube API service object
    - video_id: YouTube video ID
    - thumbnail_path: थंबनेल इमेज का पथ
    """

    try:
        # थंबनेल फ़ाइल की जाँच करें
        if not os.path.exists(thumbnail_path):
            print(f"❌ थंबनेल फ़ाइल नहीं मिली: {thumbnail_path}")
            return False

        # थंबनेल specifications check करें
        try:
            with Image.open(thumbnail_path) as img:
                width, height = img.size
                print(f"📊 थंबनेल डाइमेंशन: {width}x{height}")

                # Recommended size: 1280x720 (16:9 ratio)
                if width < 640 or height < 480:
                    print("⚠️  Warning: थंबनेल बहुत छोटा है (minimum 640x480 recommended)")

                # File format check
                if img.format not in ['JPEG', 'JPG', 'PNG', 'GIF', 'BMP']:
                    print("⚠️  Warning: YouTube केवल JPEG, PNG, GIF, BMP सपोर्ट करता है")
        except Exception as e:
            print(f"⚠️  इमेज check में error: {e}")

        # थंबनेल अपलोड करें
        print(f"🖼️  थंबनेल अपलोड हो रहा है...")

        with open(thumbnail_path, 'rb') as thumbnail_file:
            media = MediaIoBaseUpload(
                thumbnail_file,
                mimetype='image/jpeg' if thumbnail_path.lower().endswith(('.jpg', '.jpeg')) else 'image/png',
                resumable=True
            )

            request = youtube.thumbnails().set(
                videoId=video_id,
                media_body=media
            )

            response = request.execute()

            print("✅ थंबनेल सफलतापूर्वक अपलोड हुआ!")
            print(f"   Size: {response.get('size', 'N/A')}")
            print(f"   Video ID: {video_id}")

            return True

    except Exception as e:
        print(f"❌ थंबनेल अपलोड में error: {str(e)}")
        return False

In [ ]:
# @title step 3. **वीडियो अपलोड + थंबनेल Function**

def upload_video_with_thumbnail(video_path, thumbnail_path, video_details):
    """
    वीडियो और थंबनेल एक साथ अपलोड करें

    Parameters:
    - video_path: वीडियो फ़ाइल का पथ
    - thumbnail_path: थंबनेल फ़ाइल का पथ
    - video_details: dictionary with title, description, etc.
    """

    youtube = get_youtube_service()

    # पहले वीडियो अपलोड करें
    print("🎥 वीडियो अपलोड शुरू...")

    # वीडियो metadata
    body = {
        "snippet": {
            "title": video_details.get('title', 'My Video'),
            "description": video_details.get('description', ''),
            "tags": video_details.get('tags', []),
            "categoryId": video_details.get('category_id', '22')
        },
        "status": {
            "privacyStatus": video_details.get('privacy_status', 'private'),
            "selfDeclaredMadeForKids": False
        }
    }

    # वीडियो अपलोड
    try:
        media = MediaFileUpload(
            video_path,
            mimetype='video/*',
            resumable=True
        )

        request = youtube.videos().insert(
            part=",".join(body.keys()),
            body=body,
            media_body=media
        )

        video_response = request.execute()
        video_id = video_response['id']

        print(f"✅ वीडियो अपलोड हुआ! Video ID: {video_id}")
        print(f"📺 URL: https://www.youtube.com/watch?v={video_id}")

        # थंबनेल अपलोड करें
        if thumbnail_path and os.path.exists(thumbnail_path):
            # थोड़ा wait करें (YouTube processing के लिए)
            import time
            print("⏳ वीडियो processing का इंतज़ार...")
            time.sleep(10)

            # थंबनेल अपलोड
            thumbnail_success = upload_thumbnail(youtube, video_id, thumbnail_path)

            if thumbnail_success:
                print("🎉 वीडियो और थंबनेल दोनों सफलतापूर्वक अपलोड हुए!")
            else:
                print("⚠️  वीडियो अपलोड हुआ, लेकिन थंबनेल में समस्या")
        else:
            print("⚠️  थंबनेल फ़ाइल नहीं मिली, सिर्फ वीडियो अपलोड हुआ")

        return video_id

    except Exception as e:
        print(f"❌ वीडियो अपलोड में error: {str(e)}")
        return None

In [ ]:
# @title step 4. authenticate_colab() with drive backup
# Clear any existing tokens
!rm -f token.pickle 2>/dev/null || true

# %%
import os
import pickle
import json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload
from google_auth_oauthlib.flow import Flow
from google.auth.transport.requests import Request
from google.colab import files
from PIL import Image
import requests
import time


# ## 1. **Colab-Friendly Authentication (No Browser)**

def authenticate_colab():
    """
    Colab के लिए optimized authentication - manual code entry
    """
    # SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]

    SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",
    "https://www.googleapis.com/auth/userinfo.email",
    "https://www.googleapis.com/auth/userinfo.profile",
    "openid"
]


    drive_path = "/content/drive/MyDrive/readbucks/token.pickle"
    # check drive for pickel file
    if os.path.exists(drive_path):
        print('using google drive')
        try:
            with open(drive_path, 'rb') as token:
                credentials = pickle.load(token)
            # Save credentials for future
            with open('token.pickle', 'wb') as token:
                  pickle.dump(credentials, token)

            # Check if token is expired
            if credentials and credentials.expired and credentials.refresh_token:
                credentials.refresh(Request())
                print("✅ Token refreshed successfully")
                with open(drive_path, 'wb') as token:
                    pickle.dump(credentials, token)

                # Save credentials for future
                with open('token.pickle', 'wb') as token:
                    pickle.dump(credentials, token)

            if credentials and credentials.valid:
                print("✅ Using existing credentials")
                return build('youtube', 'v3', credentials=credentials)

        except Exception as e:
            print(f"⚠️ Error loading token: {e}")

  # Check for existing token
    if os.path.exists('token.pickle'):
        try:
            with open('token.pickle', 'rb') as token:
                credentials = pickle.load(token)

            # Check if token is expired
            if credentials and credentials.expired and credentials.refresh_token:
                credentials.refresh(Request())
                print("✅ Token refreshed successfully")
                with open('token.pickle', 'wb') as token:
                    pickle.dump(credentials, token)

            if credentials and credentials.valid:
                print("✅ Using existing credentials")
                return build('youtube', 'v3', credentials=credentials)

        except Exception as e:
            print(f"⚠️ Error loading token: {e}")

    # New authentication required
    print("\n" + "="*60)
    print("🔐 NEW AUTHENTICATION REQUIRED")
    print("="*60)

    config_file_path = "/content/drive/MyDrive/readbucks/readbuck_config.json"
    # Step 1: Upload credentials
    if os.path.exists(config_file_path):
        creds_file= config_file_path
    else:
        print("\n📁 STEP 1: Upload your Google API credentials JSON file")
        print("(The file you downloaded from Google Cloud Console)")

        uploaded = files.upload()

        if not uploaded:
            print("❌ No file uploaded. Please upload credentials.json")
            return None

        creds_file = list(uploaded.keys())[0]
        print(f"✅ File uploaded: {creds_file}")

    # Step 2: Create OAuth flow with out-of-band (OOB) redirect
    try:
        flow = Flow.from_client_secrets_file(
            creds_file,
            scopes=SCOPES,
            redirect_uri='urn:ietf:wg:oauth:2.0:oob'  # OOB for manual entry
        )

        # Generate authorization URL
        auth_url, _ = flow.authorization_url(
            access_type='offline',
            prompt='consent',
            include_granted_scopes='true'
        )

        # Display instructions
        print("\n" + "="*60)
        print("📋 STEP 2: Get Authorization Code")
        print("="*60)
        print("\n1. Click this link (open in new tab):")
        print(f"\n🔗 {auth_url}")
        print("\n2. Login with your Google account")
        print("3. Click 'Allow' to grant permissions")
        print("4. You will see a code on the screen")
        print("5. Copy that code and paste it below")
        print("\n" + "="*60)

        # Get code from user
        auth_code = input("\n📝 Paste the authorization code here: ").strip()

        if not auth_code:
            print("❌ No code provided")
            return None

        # Step 3: Exchange code for tokens
        print("\n🔄 STEP 3: Exchanging code for access token...")

        flow.fetch_token(code=auth_code)
        credentials = flow.credentials

        # Save credentials for future
        with open('token.pickle', 'wb') as token:
            pickle.dump(credentials, token)


        # Save  credentials for future in drive
        with open(drive_path, 'wb') as token:
            pickle.dump(credentials, token)


        print("✅ Authentication successful!")
        print("✅ Token saved for future use")

        return build('youtube', 'v3', credentials=credentials)

    except Exception as e:
        print(f"❌ Authentication failed: {str(e)}")
        return None


# # ## 2. **Test Authentication**


# # Test authentication
# print("Testing authentication...")
# youtube = authenticate_colab()

# if youtube:
#     print("\n🎉 Authentication successful! YouTube API ready to use.")
# else:
#     print("\n❌ Authentication failed. Please check your credentials.")

In [ ]:
# @title step 5: entry point
def main():
    """
    मुख्य function - यूज़र के लिए interactive
    """
    print("="*60)
    print("YouTube Video & Thumbnail Uploader")
    print("="*60)

    try:
        # YouTube service initialize करें
        # youtube = get_youtube_service()
        youtube = authenticate_colab()


        #created above
        video_file="final_story_video.mp4"

        if not video_file:
             print("❌ कोई वीडियो फ़ाइल अपलोड नहीं हुई")
             return

        # check thumbnail image
        if thumbnail_image_file_path.exists():
            thumbnail_file=thumbnail_image_file
        else:
          print("\n🖼️  थंबनेल फ़ाइल अपलोड करें:")
          thumbnail_uploaded = files.upload()
          thumbnail_file = list(thumbnail_uploaded.keys())[0] if thumbnail_uploaded else None


        if video_details_file_path.exists():
            video_details_file = videodetails_file
        else:

            print('upload video details file (.json)')
            details_upload = files.upload()
            video_details_file = list(details_upload.keys())[0] if details_upload else None
        import json
        with open(video_details_file, "r", encoding="utf-8") as f:
              video_details = json.load(f)

            # Upload video with thumbnail

        video_id = upload_video_with_thumbnail(video_file, thumbnail_file, video_details)


    except Exception as e:
        print(f"\n❌ Error: {str(e)}")



# start execution here
if __name__ == "__main__":
    main()

# **YouTube Shorts Video Creation and Upload**

In [ ]:
# @title create shorts audio file (shorts_story.mp3)
import edge_tts

SHORTS_TEXT = shorts_text_to_speak
SHORTS_VOICE = "hi-IN-SwaraNeural" # Same voice as long video
SHORTS_OUTPUT_AUDIO_FILE = "shorts_story.mp3"

# रफ़्तार (Speed), पिच (Pitch), और वॉल्यूम (Volume) सेट करना
SHORTS_SPEED = "+50%"    # Slightly faster for shorts
SHORTS_PITCH = "+0Hz"
SHORTS_VOLUME = "+200%"

async def main_shorts_audio():
    communicate = edge_tts.Communicate(
        SHORTS_TEXT,
        SHORTS_VOICE,
        rate=SHORTS_SPEED,
        pitch=SHORTS_PITCH,
        volume=SHORTS_VOLUME
    )
    await communicate.save(SHORTS_OUTPUT_AUDIO_FILE)

await main_shorts_audio()
print(f"Shorts audio updated with Speed: {SHORTS_SPEED}, Pitch: {SHORTS_PITCH}, Volume: {SHORTS_VOLUME}")

### Create Shorts Video (final_shorts_video.mp4) using `ShortsVideoCreator`

In [ ]:
# @title ShortsVideoCreator class
import numpy as np
import cv2
from moviepy.editor import *
import moviepy.audio.fx.all as afx
from PIL import Image, ImageFilter
import os
import tempfile
import random
from tqdm.auto import tqdm

class ShortsVideoCreator:
    def __init__(self, image_path, audio_path, output_path="final_shorts_video.mp4"):
        self.image_path = image_path
        self.audio_path = audio_path
        self.output_path = output_path

        # Shorts dimensions (9:16 aspect ratio)
        self.width = 720  # Standard width for vertical video
        self.height = 1280 # Standard height for vertical video

        audio_clip = AudioFileClip(audio_path)
        self.sample_rate = audio_clip.fps
        self.duration = audio_clip.duration
        chunks = list(audio_clip.iter_chunks(fps=self.sample_rate, chunksize=100000))
        self.audio_data = np.concatenate(chunks)
        if len(self.audio_data.shape) > 1:
            self.audio_data = np.mean(self.audio_data, axis=1)
        audio_clip.close()

        # Prepare the image for 9:16 shorts
        self.original_image = self._prepare_image_for_shorts(image_path)
        # Particles for effects
        self.particles = [{'x': random.random(), 'y': random.random(), 's': random.random()} for _ in range(50)]

    def _prepare_image_for_shorts(self, image_path):
        img = Image.open(image_path).convert('RGB')
        orig_width, orig_height = img.size

        target_aspect = self.width / self.height
        image_aspect = orig_width / orig_height

        if image_aspect > target_aspect:
            # Image is wider than target aspect, crop width
            new_width = int(orig_height * target_aspect)
            left = (orig_width - new_width) / 2
            right = (orig_width + new_width) / 2
            top, bottom = 0, orig_height
            img = img.crop((left, top, right, bottom))
        elif image_aspect < target_aspect:
            # Image is taller than target aspect, crop height
            new_height = int(orig_width / target_aspect)
            top = (orig_height - new_height) / 2
            bottom = (orig_height + new_height) / 2
            left, right = 0, orig_width
            img = img.crop((left, top, right, bottom))

        # Resize to target dimensions
        img = img.resize((self.width, self.height), Image.Resampling.LANCZOS)
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

    def get_smooth_amplitude(self, duration, total_frames):
        samples_per_frame = len(self.audio_data) // total_frames
        envelope = []
        for i in range(total_frames):
            start, end = i * samples_per_frame, (i + 1) * samples_per_frame
            amp = np.max(np.abs(self.audio_data[start:end])) if start < len(self.audio_data) else 0
            envelope.append(amp)
        if max(envelope) > 0: envelope = np.array(envelope) / max(envelope)
        return np.convolve(envelope, np.ones(5)/5, mode='same')

    def apply_shorts_effects(self, img, amp, frame_num, total_frames):
        # Effects adapted for 9:16 vertical video
        h, w = img.shape[:2] # h=1280, w=720
        t = frame_num / total_frames

        # Zoom and subtle pan
        zoom_factor = 1.0 + 0.05 * np.sin(t * np.pi * 2) # More subtle zoom
        pan_x = int(np.sin(t * 1.5) * 10) # Smaller pan
        pan_y = int(np.cos(t * 1.5) * 10)
        M = np.float32([[zoom_factor, 0, (1-zoom_factor)*w/2 + pan_x], [0, zoom_factor, (1-zoom_factor)*h/2 + pan_y]])
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

        # Particle overlay (falling elements - adjusted for vertical)
        overlay = img.copy()
        for p in self.particles:
            px = int(p['x'] * w)
            # Particles fall down, wrap around at the bottom
            py = int((p['y'] + (frame_num * 0.005 * p['s'])) % 1 * h) # Faster fall for shorts
            size = int(1 + amp * 3 * p['s']) # Smaller particles
            cv2.circle(overlay, (px, py), size, (255, 255, 255), -1)
        img = cv2.addWeighted(img, 0.9, overlay, 0.1, 0) # Less intense overlay

        # Audio wave bars at the bottom, similar to long video
        wave_height = 100 # Max height of bars
        num_bars = 60 # Number of bars
        bar_width = w // num_bars
        # Offset from bottom
        bottom_offset = 30
        for i in range(num_bars):
            dist_from_center = 1.0 - abs(i - num_bars/2) / (num_bars/2)
            h_val = int(wave_height * amp * dist_from_center * np.random.uniform(0.6, 1.0))
            color = (int(100 + 155*t), int(200 - 100*amp), 255) # Dynamic color
            # Draw lines from bottom upwards
            cv2.line(img, (i*bar_width + bar_width // 2, h - bottom_offset),
                     (i*bar_width + bar_width // 2, h - bottom_offset - h_val),
                     color, 2, cv2.LINE_AA)

        return img

    def create_video(self, fps=24):
        total_frames = int(self.duration * fps)
        envelope = self.get_smooth_amplitude(self.duration, total_frames)
        temp_video = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        temp_video.close()

        out = cv2.VideoWriter(temp_video.name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (self.width, self.height))

        print("Rendering Shorts Visual Frames...")
        for i in tqdm(range(total_frames), desc="Shorts Video Rendering"):
            frame = self.apply_shorts_effects(self.original_image.copy(), envelope[i], i, total_frames)
            out.write(frame)
        out.release()

        print("Finalizing Shorts Audio and Exporting...")
        final_clip = VideoFileClip(temp_video.name)
        voice_audio = AudioFileClip(self.audio_path)
        final_clip = final_clip.set_audio(voice_audio)

        final_clip.write_videofile(self.output_path, codec='libx264', audio_codec='aac', bitrate="3000k", logger='bar') # Higher bitrate for shorts
        os.unlink(temp_video.name)

In [ ]:
# @title create shorts video (final_shorts_video.mp4)

def main_shorts_video():
    shorts_image_path = shorts_bg_image_file_path # Uploaded in a previous step
    shorts_audio_path = SHORTS_OUTPUT_AUDIO_FILE # Created in a previous step
    shorts_output_path = "final_shorts_video.mp4"

    if not os.path.exists(shorts_image_path):
        print(f"❌ Error: Shorts background image not found at {shorts_image_path}.")
        return
    if not os.path.exists(shorts_audio_path):
        print(f"❌ Error: Shorts audio file not found at {shorts_audio_path}.")
        return

    print("Starting Shorts video creation...")
    try:
        shorts_creator = ShortsVideoCreator(str(shorts_image_path), shorts_audio_path, shorts_output_path)
        shorts_creator.create_video()
        print(f"✅ Shorts video successfully created at: {shorts_output_path}")
    except Exception as e:
        print(f"❌ Error during Shorts video creation: {e}")

if __name__ == "__main__":
    main_shorts_video()

In [ ]:
# @title upload shorts video to YouTube

def upload_shorts_video_to_youtube():
    print("="*60)
    print("YouTube Shorts Video Uploader")
    print("="*60)

    try:
        # Authenticate with YouTube API (reusing existing auth)
        youtube_service = authenticate_colab()
        SHORTS_OUTPUT_VIDEO_PATH ="final_shorts_video.mp4"
        shorts_video_file = SHORTS_OUTPUT_VIDEO_PATH # From previous step

        if not os.path.exists(shorts_video_file):
            print(f"❌ Shorts video file not found at {shorts_video_file}. Cannot upload.")
            return

        # Reuse the thumbnail uploaded for the long video, or ask for a new one if preferred.
        # For now, using the last uploaded thumbnail.
        shorts_thumbnail_path = thumbnail_image_file_path # from cell hgId9koHaWV7
        if not os.path.exists(shorts_thumbnail_path):
            print("⚠️ Thumbnail file not found. Please upload a thumbnail for the short video if you want one.")
            shorts_thumbnail_path = None # Do not upload thumbnail if not found

        # Upload shorts video with its details and thumbnail
        print("🚀 Uploading Shorts Video to YouTube...")
        video_id = upload_video_with_thumbnail(
            shorts_video_file,
            shorts_thumbnail_path,
            shorts_video_details # Loaded from shorts_video_details_file_path
        )

        if video_id:
            print(f"✅ Shorts Video uploaded successfully! YouTube ID: {video_id}")
            print(f"🔗 Watch here: https://www.youtube.com/watch?v={video_id}")
        else:
            print("❌ Failed to upload Shorts Video.")

    except Exception as e:
        print(f"❌ Error during Shorts video upload process: {str(e)}")

# Call the function to start the upload process
upload_shorts_video_to_youtube()

In [ ]:
print("कृपया audio story फ़ाइल अपलोड करें (txt)")
SHORTS_OUTPUT_AUDIO_FILE = "shorts_story.mp3"
audio = files.upload()
audio_filename = list(audio.keys())[0] if audio else None
audio_story_file_path = Path(audio_filename)